# Gold Price Prediction — Exploration

Exploratory analysis for the gold price nowcast model. All logic lives in `src/` — this notebook only imports and visualizes.

**Key finding preserved here for honesty:** the original version of this project reported 98.94% R² using a *random* train/test split. That number was inflated by data leakage — random splits let the model train on days adjacent to its test days. With a proper chronological split the raw-price model scores a *negative* R². The fix (predicting returns instead of price levels) is evaluated at the bottom.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score

from src.data_loader import load_data, time_based_split
from src.features import MARKET_COLUMNS, build_features
from src.model import GoldPriceModel

df = load_data('../data/gld_price_data.csv')
df.info()

## Correlation analysis

At the level of raw prices, silver (SLV) is the only feature strongly correlated with gold. SPX and EUR/USD are nearly uncorrelated over the full 2008–2018 window.

In [ ]:
correlation = df.drop(columns=['Date']).corr()
plt.figure(figsize=(8, 8))
sns.heatmap(correlation, square=True, annot=True, cbar=True, cmap='coolwarm')
plt.title('Correlation of raw price levels')
plt.show()

correlation['GLD'].sort_values(ascending=False)

In [ ]:
sns.histplot(df['GLD'], kde=True)
plt.title('Distribution of GLD prices, 2008\u20132018')
plt.xlabel('GLD price ($)')
plt.show()

## The leakage problem

Same Random Forest on raw price levels, two different splits. The random split reproduces the original ~98.9% number; the chronological split shows the model actually cannot generalize forward in time from raw levels (trees cannot extrapolate beyond the price range they were trained on).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X, y = df[MARKET_COLUMNS], df['GLD']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
leaky = RandomForestRegressor(random_state=42).fit(Xtr, ytr)
print(f'Random split (leaky):  R2 = {r2_score(yte, leaky.predict(Xte)):.4f}')

train_df, test_df = time_based_split(df)
honest = RandomForestRegressor(random_state=42).fit(train_df[MARKET_COLUMNS], train_df['GLD'])
pred = honest.predict(test_df[MARKET_COLUMNS])
print(f'Time split (honest):   R2 = {r2_score(test_df["GLD"], pred):.4f}  MAE = ${mean_absolute_error(test_df["GLD"], pred):.2f}')

## The fixed model: return space + lag/rolling features

The shipped model predicts GLD's **same-day return** from same-day moves in SPX/USO/SLV/EUR-USD plus short-term context, then reconstructs the price from yesterday's close. Evaluated on the same chronological split, and compared against the naive persistence baseline ("predict yesterday's price").

In [ ]:
features = build_features(df)
train_f, test_f = time_based_split(features)

model = GoldPriceModel().train(train_f, train_f['target_ret'])
pred_price = model.predict_price(test_f, test_f['GLD_lag1'])
naive = test_f['GLD_lag1']

print(f'Final model:  R2 = {r2_score(test_f["GLD"], pred_price):.4f}  MAE = ${mean_absolute_error(test_f["GLD"], pred_price):.2f}')
print(f'Naive:        R2 = {r2_score(test_f["GLD"], naive):.4f}  MAE = ${mean_absolute_error(test_f["GLD"], naive):.2f}')

plt.figure(figsize=(12, 5))
plt.plot(test_f['Date'], test_f['GLD'], label='Actual', color='blue', linewidth=1)
plt.plot(test_f['Date'], pred_price, label='Predicted', color='red', linewidth=1, alpha=0.7)
plt.legend()
plt.title('Actual vs predicted GLD, chronological test period')
plt.ylabel('GLD price ($)')
plt.show()

In [ ]:
model.feature_importance().sort_values().plot.barh(figsize=(8, 5), color='goldenrod')
plt.title('Feature importance')
plt.xlabel('Importance')
plt.show()

The same-day silver move (`SLV_ret1`) dominates, which matches financial intuition: gold and silver are both precious metals driven by the same macro forces (real rates, USD strength, risk sentiment), so they co-move tightly within a day.